In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img

# fcontrast - test moral vs non-moral for each story separately

In [11]:
virtueORvice = 'vices' # vices, virtues



fcontrast_dir = f"/Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/{virtueORvice}/fcontrast/"

# stories = ["21styear","black","bronx","forgot","lucy","notthefallintact",
#            "pieman","piemanpni","shapesphysical","shapessocial","slumlordreach","tunnel"]

stories = ["tunnel"]


In [12]:

for story in stories:
    print(f"\nProcessing story: {story}")
    subfolder = os.path.join(fcontrast_dir, story)
    files = [f for f in os.listdir(subfolder) if f.endswith(".nii") or f.endswith(".nii.gz")]
    full_paths = [os.path.join(subfolder, f) for f in files]
    
    # Sanity check
    if not full_paths:
        print(f"No .nii or .nii.gz files found in {subfolder}. Skipping...")
        continue
    
    # Create design matrix
    design_matrix = pd.DataFrame(
        [1] * len(full_paths),
        columns=["intercept"],
    )

    # Fit the second-level model
    second_level_model = SecondLevelModel()
    second_level_model = second_level_model.fit(
        full_paths,
        design_matrix=design_matrix,
    )

    # Compute z-map
    z_map = second_level_model.compute_contrast(
        second_level_contrast="intercept",
        output_type="z_score",
    )

    # Save z-map in subfolder
    z_map_filename = os.path.join(subfolder, "zscores_fcontrast_secondlevel.nii.gz")
    z_map.to_filename(z_map_filename)
    print(f"✅ Saved z-map to: {z_map_filename}")

    # Threshold map
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='bonferroni', #bonferroni,fdr
        cluster_threshold=0, #2,5,10,0
        two_sided=True,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(
        subfolder,
        f"threshold_{threshold:.3g}_bonfcorrect_zscores_fcontrast_secondlevel.nii.gz",
    )
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")




Processing story: tunnel
✅ Saved z-map to: /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/vices/fcontrast/tunnel/zscores_fcontrast_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/vices/fcontrast/tunnel/threshold_4.91_bonfcorrect_zscores_fcontrast_secondlevel.nii.gz


In [6]:
# # Collect z-maps
# z_maps = []
# for story in stories:
#     z_map_path = os.path.join(fcontrast_dir, story, "zscores_fcontrast_secondlevel.nii.gz")
#     if os.path.exists(z_map_path):
#         print(f"✅ Loaded {z_map_path}")
#         z_maps.append(image.load_img(z_map_path))
#     else:
#         print(f"⚠️ Missing: {z_map_path}")

# if len(z_maps) < 2:
#     raise ValueError("Need at least two z-maps to compute a minimum-statistic conjunction.")

# # Resample all to the first image’s space
# target_img = z_maps[0]
# z_maps_resampled = [image.resample_to_img(img, target_img, interpolation="continuous") for img in z_maps]

# # Create dictionary dynamically
# img_dict = {f"img{i+1}": z_maps_resampled[i] for i in range(len(z_maps_resampled))}
# expr = "np.minimum.reduce((" + ", ".join(img_dict.keys()) + "))"

# # Compute conjunction map
# min_stat_map = image.math_img(expr, **img_dict)

# # Save output
# output_path = os.path.join(fcontrast_dir, "minimum_stat_conjunction.nii.gz")
# min_stat_map.to_filename(output_path)
# print(f"✅ Saved conjunction map to: {output_path}")


✅ Loaded /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/21styear/zscores_fcontrast_secondlevel.nii.gz
✅ Loaded /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/black/zscores_fcontrast_secondlevel.nii.gz
✅ Loaded /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/bronx/zscores_fcontrast_secondlevel.nii.gz
✅ Loaded /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/forgot/zscores_fcontrast_secondlevel.nii.gz
✅ Loaded /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/lucy/zscores_fcontrast_secondlevel.nii.gz
✅ Loaded /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/notthefallintact/zscores_fcontrast_secondlevel.nii.gz
✅ Loaded /Users/silvycolli

/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_6327/1583833227.py:16: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  z_maps_resampled = [image.resample_to_img(img, target_img, interpolation="continuous") for img in z_maps]
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/image/resampling.py:805: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  return resample_img(
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_6327/1583833227.py:16: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  z_maps_resampled = [image.resample_to_img(img, target_img, interpolation="continuou

✅ Saved conjunction map to: /Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/minimum_stat_conjunction.nii.gz


In [9]:

thresholded_map, threshold = threshold_stats_img(
    stat_img=min_stat_map,
    alpha=0.05,            # FDR level (start here)
    height_control='fdr',  # use 'bonferroni' for more conservative FWER control
    cluster_threshold=0,   # use >0 to filter small clusters
    two_sided=False        # or True if you want two-sided test
)

print("FDR threshold (stat):", threshold)
thresholded_map.to_filename("min_stat_conjunction_fdrcorrected.nii.gz")

FDR threshold (stat): inf
